# Combine GEE Features and PM25 Data

This notebook merges two CSV files:
- **gee_features_daily.csv**: Environmental features (temperature, pressure, wind, NO2, CO, O3, AOD)
- **pm25.csv**: PM2.5 air quality measurements and sensor locations

The merge is performed on three key fields:
- `date`: Calendar date
- `latitude`: Geographic latitude coordinate
- `longitude`: Geographic longitude coordinate

Output: **combined_gee_pm25.csv** containing all features and PM2.5 levels for each location on each date.

## Step 0: Import Libraries and Setup Paths

In [1]:
import pandas as pd
from pathlib import Path

# Get the base directory (workspace root)
notebook_dir = Path.cwd()
# If running from scripts folder, go up one level
if notebook_dir.name == 'notebooks':
    base_dir = notebook_dir.parent
else:
    base_dir = notebook_dir

data_dir = base_dir / 'data'

print(f"Base directory: {base_dir}")
print(f"Data directory: {data_dir}")
print(f"\nData files present:")
for file in data_dir.glob('*.csv'):
    print(f"  - {file.name}")

Base directory: c:\Users\djame\OneDrive\Bureau\3rd Year\Semester 2\Group Project\Global_approach
Data directory: c:\Users\djame\OneDrive\Bureau\3rd Year\Semester 2\Group Project\Global_approach\data

Data files present:
  - combined_gee_pm25.csv
  - final_dataset_with_landuse.csv
  - gee_features_daily.csv
  - GlobalWeatherRepository.csv
  - land_features.csv


## Step 1: Load Data Files

In [2]:
print("Loading data files...\n")

# Load the CSV files
pm25_df = pd.read_csv(data_dir / 'GlobalWeatherRepository.csv')
gee_df = pd.read_csv(data_dir / 'gee_features_daily.csv')

print(f"PM25 data loaded: {pm25_df.shape[0]} rows × {pm25_df.shape[1]} columns")
print(f"GEE data loaded: {gee_df.shape[0]} rows × {gee_df.shape[1]} columns")

print(f"\nPM25 columns: {list(pm25_df.columns)}")
print(f"\nGEE columns: {list(gee_df.columns)}")

Loading data files...

PM25 data loaded: 139168 rows × 41 columns
GEE data loaded: 186013 rows × 5 columns

PM25 columns: ['country', 'location_name', 'latitude', 'longitude', 'timezone', 'last_updated_epoch', 'last_updated', 'temperature_celsius', 'temperature_fahrenheit', 'condition_text', 'wind_mph', 'wind_kph', 'wind_degree', 'wind_direction', 'pressure_mb', 'pressure_in', 'precip_mm', 'precip_in', 'humidity', 'cloud', 'feels_like_celsius', 'feels_like_fahrenheit', 'visibility_km', 'visibility_miles', 'uv_index', 'gust_mph', 'gust_kph', 'air_quality_Carbon_Monoxide', 'air_quality_Ozone', 'air_quality_Nitrogen_dioxide', 'air_quality_Sulphur_dioxide', 'air_quality_PM2.5', 'air_quality_PM10', 'air_quality_us-epa-index', 'air_quality_gb-defra-index', 'sunrise', 'sunset', 'moonrise', 'moonset', 'moon_phase', 'moon_illumination']

GEE columns: ['date', 'id', 'latitude', 'longitude', 'AOD']


## Step 2: Preview Data

In [3]:
print("First few rows of PM25 data:")
print(pm25_df.head())


print("\nFirst few rows of GEE data:")
print(gee_df.head())

First few rows of PM25 data:
       country     location_name  latitude  longitude        timezone  \
0  Afghanistan             Kabul     34.52      69.18      Asia/Kabul   
1      Albania            Tirana     41.33      19.82   Europe/Tirane   
2      Algeria           Algiers     36.76       3.05  Africa/Algiers   
3      Andorra  Andorra La Vella     42.50       1.52  Europe/Andorra   
4       Angola            Luanda     -8.84      13.23   Africa/Luanda   

   last_updated_epoch      last_updated  temperature_celsius  \
0          1715849100  2024-05-16 13:15                 26.6   
1          1715849100  2024-05-16 10:45                 19.0   
2          1715849100  2024-05-16 09:45                 23.0   
3          1715849100  2024-05-16 10:45                  6.3   
4          1715849100  2024-05-16 09:45                 26.0   

   temperature_fahrenheit condition_text  ...  air_quality_PM2.5  \
0                    79.8  Partly Cloudy  ...                8.4   
1          

In [4]:
print(pm25_df.shape)
print(gee_df.shape)

(139168, 41)
(186013, 5)


In [5]:
pm25_df['air_quality_PM2.5']

0           8.40
1           1.10
2          10.40
3           0.70
4         183.40
           ...  
139163     13.75
139164     66.55
139165      8.85
139166      7.55
139167     16.45
Name: air_quality_PM2.5, Length: 139168, dtype: float64

## Step 3: Standardize Column Names

In [6]:
print("Standardizing column names...")
print(f"Original GEE columns: {list(gee_df.columns)}")

# Rename lat/lon to latitude/longitude in GEE data for consistency
gee_df = gee_df.rename(columns={'lat': 'latitude', 'lon': 'longitude'})
pm25_df = pm25_df.rename(columns={'lat': 'latitude', 'lon': 'longitude' ,'air_quality_PM2.5' : 'pm25' , 'last_updated' : 'date'})

print(f"After rename: {list(gee_df.columns)}")
print(f"After rename: {list(pm25_df.columns)}")

Standardizing column names...
Original GEE columns: ['date', 'id', 'latitude', 'longitude', 'AOD']
After rename: ['date', 'id', 'latitude', 'longitude', 'AOD']
After rename: ['country', 'location_name', 'latitude', 'longitude', 'timezone', 'last_updated_epoch', 'date', 'temperature_celsius', 'temperature_fahrenheit', 'condition_text', 'wind_mph', 'wind_kph', 'wind_degree', 'wind_direction', 'pressure_mb', 'pressure_in', 'precip_mm', 'precip_in', 'humidity', 'cloud', 'feels_like_celsius', 'feels_like_fahrenheit', 'visibility_km', 'visibility_miles', 'uv_index', 'gust_mph', 'gust_kph', 'air_quality_Carbon_Monoxide', 'air_quality_Ozone', 'air_quality_Nitrogen_dioxide', 'air_quality_Sulphur_dioxide', 'pm25', 'air_quality_PM10', 'air_quality_us-epa-index', 'air_quality_gb-defra-index', 'sunrise', 'sunset', 'moonrise', 'moonset', 'moon_phase', 'moon_illumination']


## Step 4: Convert Date Columns to Datetime Format

In [7]:
print("Converting date columns to datetime format...\n")

pm25_df['date'] = pd.to_datetime(pm25_df['date'], format='mixed', errors='coerce').dt.normalize()
gee_df['date'] = pd.to_datetime(gee_df['date'], format='mixed', errors='coerce').dt.normalize()

if pm25_df['date'].isna().any():
    print(f"Warning: {pm25_df['date'].isna().sum()} PM25 rows could not be parsed as dates.")
if gee_df['date'].isna().any():
    print(f"Warning: {gee_df['date'].isna().sum()} GEE rows could not be parsed as dates.")

print(f"PM25 date range: {pm25_df['date'].min().date()} to {pm25_df['date'].max().date()}")
print(f"GEE date range: {gee_df['date'].min().date()} to {gee_df['date'].max().date()}")

print(f"\nPM25: {pm25_df['date'].nunique()} unique dates")
print(f"GEE: {gee_df['date'].nunique()} unique dates")

Converting date columns to datetime format...

PM25 date range: 2024-05-16 to 2026-05-03
GEE date range: 2024-05-16 to 2026-05-01

PM25: 717 unique dates
GEE: 715 unique dates


## Step 5: Merge Datasets on Date, Latitude, and Longitude

# SKIP

In [34]:
gee_locations = gee_df.groupby(['latitude', 'longitude', 'date']).mean().reset_index()[['latitude', 'longitude', 'date']]

gee_locations.head()

,latitude,longitude,date
0,-41.3,174.78,2024-05-16
1,-41.3,174.78,2024-05-17
2,-41.3,174.78,2024-05-18
3,-41.3,174.78,2024-05-24
4,-41.3,174.78,2024-05-25


In [35]:
gee_locations.shape

(186013, 3)

In [ ]:
pm25_locations = pm25_df.groupby(['latitude', 'longitude', 'date'])['pm25'].mean().reset_index()[['latitude', 'longitude', 'date']]

pm25_locations.head()

,latitude,longitude,date
0,-41.3,174.78,2024-05-16
1,-41.3,174.78,2024-05-17
2,-41.3,174.78,2024-05-18
3,-41.3,174.78,2024-05-19
4,-41.3,174.78,2024-05-20


In [38]:
pm25_locations.shape

(138918, 3)

In [40]:
merge = pd.merge(gee_locations, pm25_locations, on=['latitude', 'longitude', 'date'], how='inner')

merge.head()

,latitude,longitude,date
0,-41.3,174.78,2024-05-16
1,-41.3,174.78,2024-05-17
2,-41.3,174.78,2024-05-18
3,-41.3,174.78,2024-05-24
4,-41.3,174.78,2024-05-25


In [42]:
# Based on a key column 'id'
pm25_locations[~pm25_locations[['latitude', 'longitude', 'date']].isin(merge[['latitude', 'longitude', 'date']]).any(axis=1)].head()

,latitude,longitude,date
494,-41.3,174.7833,2025-09-30
495,-41.3,174.7833,2025-10-01
496,-41.3,174.7833,2025-10-02
497,-41.3,174.7833,2025-10-03
498,-41.3,174.7833,2025-10-04


In [57]:
pm25_df[pm25_df['date'] == '2025-11-30']

,country,location_name,latitude,longitude,timezone,last_updated_epoch,date,temperature_celsius,temperature_fahrenheit,condition_text,...,pm25,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination
109328,Afghanistan,Kabul,34.5167,69.1833,Asia/Kabul,1764485100,2025-11-30,11.2,52.1,Sunny,...,19.55,20.25,2,2,06:41 AM,04:43 PM,01:21 PM,01:05 AM,Waxing Gibbous,66
109329,Albania,Tirana,41.3275,19.8189,Europe/Tirane,1764485100,2025-11-30,8.0,46.4,Fog,...,17.45,20.05,2,2,06:46 AM,04:12 PM,01:09 PM,01:02 AM,Waxing Gibbous,68
109330,Algeria,Algiers,36.7631,3.0506,Africa/Algiers,1764485100,2025-11-30,4.0,39.2,Clear,...,8.55,9.15,1,1,07:41 AM,05:32 PM,02:19 PM,02:12 AM,Waxing Gibbous,68
109331,Andorra,Andorra La Vella,42.5000,1.5167,Europe/Andorra,1764485100,2025-11-30,-3.0,26.7,Partly Cloudy,...,1.85,2.15,1,1,08:03 AM,05:22 PM,02:22 PM,02:19 AM,Waxing Gibbous,68
109332,Angola,Luanda,-8.8383,13.2344,Africa/Luanda,1764485100,2025-11-30,25.2,77.4,Overcast,...,12.05,16.75,1,2,05:38 AM,06:14 PM,01:54 PM,01:26 AM,Waxing Gibbous,68
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109519,Vietnam,Hanoi,21.0333,105.8500,Asia/Bangkok,1764486000,2025-11-30,26.3,79.3,Partly cloudy,...,120.55,120.95,4,10,06:16 AM,05:14 PM,01:24 PM,01:03 AM,Waxing Gibbous,65
109520,Yemen,Sanaa,15.3547,44.2067,Asia/Aden,1764486000,2025-11-30,13.2,55.8,Sunny,...,21.15,27.05,2,2,06:13 AM,05:31 PM,01:39 PM,01:19 AM,Waxing Gibbous,67
109521,Zambia,Lusaka,-15.4167,28.2833,Africa/Lusaka,1764486000,2025-11-30,24.7,76.4,Partly Cloudy,...,9.25,9.35,1,1,05:27 AM,06:24 PM,01:54 PM,01:24 AM,Waxing Gibbous,67
109522,Zimbabwe,Harare,-17.8178,31.0447,Africa/Harare,1764486000,2025-11-30,23.1,73.6,Sunny,...,20.45,20.65,2,2,05:11 AM,06:18 PM,01:43 PM,01:12 AM,Waxing Gibbous,67


In [58]:
gee_df[gee_df['date'] == '2025-11-30']

,date,id,latitude,longitude,AOD
148491,2025-11-30,0,-41.3000,174.7800,0.1290
148492,2025-11-30,1,-41.3000,174.7833,0.1290
148493,2025-11-30,3,-35.2833,149.2167,0.1710
148494,2025-11-30,4,-35.2800,149.2200,0.1710
148495,2025-11-30,9,-33.4500,-70.6700,0.0555
...,...,...,...,...,...
148748,2025-11-30,406,51.5200,-0.1100,0.0300
148749,2025-11-30,409,52.3700,4.8900,0.0420
148750,2025-11-30,410,52.3740,4.8897,0.0420
148751,2025-11-30,413,53.3300,-6.2500,0.1640


In [59]:
pm25_df['pm25'].describe()

count    139168.000000
mean         24.019831
std          36.525725
min           0.168000
25%           7.050000
50%          14.060000
75%          27.565000
max        1614.100000
Name: pm25, dtype: float64

# RESUME

In [10]:
print("Merging datasets on date, latitude, and longitude...\n")
print(f"Merge type: Inner join (only matching rows in both datasets)")
print(f"Key fields: date, latitude, longitude\n")

merged_df = pd.merge(
    gee_df,
    pm25_df,
    on=['date', 'latitude', 'longitude'],
    how='inner'
)

print(f"✓ Merge completed successfully!")
print(f"Merged data shape: {merged_df.shape[0]} rows × {merged_df.shape[1]} columns")
print(f"\nColumns in merged dataset ({merged_df.shape[1]}):")
for i, col in enumerate(merged_df.columns, 1):
    print(f"  {i:2d}. {col}")

Merging datasets on date, latitude, and longitude...

Merge type: Inner join (only matching rows in both datasets)
Key fields: date, latitude, longitude



✓ Merge completed successfully!
Merged data shape: 83704 rows × 43 columns

Columns in merged dataset (43):
   1. date
   2. id
   3. latitude
   4. longitude
   5. AOD
   6. country
   7. location_name
   8. timezone
   9. last_updated_epoch
  10. temperature_celsius
  11. temperature_fahrenheit
  12. condition_text
  13. wind_mph
  14. wind_kph
  15. wind_degree
  16. wind_direction
  17. pressure_mb
  18. pressure_in
  19. precip_mm
  20. precip_in
  21. humidity
  22. cloud
  23. feels_like_celsius
  24. feels_like_fahrenheit
  25. visibility_km
  26. visibility_miles
  27. uv_index
  28. gust_mph
  29. gust_kph
  30. air_quality_Carbon_Monoxide
  31. air_quality_Ozone
  32. air_quality_Nitrogen_dioxide
  33. air_quality_Sulphur_dioxide
  34. pm25
  35. air_quality_PM10
  36. air_quality_us-epa-index
  37. air_quality_gb-defra-index
  38. sunrise
  39. sunset
  40. moonrise
  41. moonset
  42. moon_phase
  43. moon_illumination


## Step 6: Sort Data for Readability

In [11]:
print("Sorting data by date and coordinates...\n")

merged_df = merged_df.sort_values(['date', 'latitude', 'longitude']).reset_index(drop=True)

print("✓ Data sorted by date, latitude, longitude")
print(f"\nFirst few rows of merged data:")
print(merged_df.head())

Sorting data by date and coordinates...

✓ Data sorted by date, latitude, longitude

First few rows of merged data:
        date  id  latitude  longitude       AOD      country location_name  \
0 2024-05-16   0    -41.30     174.78  0.149667  New Zealand    Wellington   
1 2024-05-16   4    -35.28     149.22  0.120500    Australia      Canberra   
2 2024-05-16   7    -34.59     -58.67  0.020000    Argentina  Buenos Aires   
3 2024-05-16   7    -34.59     -58.67  0.020000    Argentina  Buenos Aires   
4 2024-05-16   9    -33.45     -70.67  0.178500        Chile      Santiago   

                         timezone  last_updated_epoch  temperature_celsius  \
0                Pacific/Auckland          1715849100                 13.0   
1                Australia/Sydney          1715849100                  9.0   
2  America/Argentina/Buenos_Aires          1715849100                  8.0   
3  America/Argentina/Buenos_Aires          1715868000                 11.0   
4                America/

## Step 7: Save Combined Dataset

In [12]:
print(data_dir)


c:\Users\djame\OneDrive\Bureau\3rd Year\Semester 2\Group Project\Global_approach\data


In [13]:
print("Saving combined dataset...\n")

output_path = data_dir / 'combined_gee_pm25.csv'
merged_df.to_csv(output_path, index=False)

print(f"✓ Output saved to: {output_path}")
print(f"\nFile size: {output_path.stat().st_size / 1024:.2f} KB")

Saving combined dataset...

✓ Output saved to: c:\Users\djame\OneDrive\Bureau\3rd Year\Semester 2\Group Project\Global_approach\data\combined_gee_pm25.csv

File size: 21928.52 KB


## Summary and Verification

In [14]:
print("="*70)
print("MERGE SUMMARY")
print("="*70)

print(f"\nInput files:")
print(f"  PM25:                {pm25_df.shape[0]:,} rows")
print(f"  GEE Features:        {gee_df.shape[0]:,} rows")

print(f"\nOutput file (combined_gee_pm25.csv):")
print(f"  Rows:                {merged_df.shape[0]:,}")
print(f"  Columns:             {merged_df.shape[1]}")

print(f"\nData coverage:")
print(f"  Date range:          {merged_df['date'].min().date()} to {merged_df['date'].max().date()}")
print(f"  Unique dates:        {merged_df['date'].nunique()}")
print(f"  Unique locations:    {len(merged_df[['latitude', 'longitude']].drop_duplicates())}")

print(f"\nMissing values per column:")
missing = merged_df.isnull().sum()
missing_cols = missing[missing > 0]
if len(missing_cols) == 0:
    print("  ✓ No missing values!")
else:
    for col, count in missing_cols.items():
        pct = (count / len(merged_df)) * 100
        print(f"  {col}: {count:,} ({pct:.1f}%)")

print("\n" + "="*70)
print("✓ Merge completed successfully!")
print("="*70)

MERGE SUMMARY

Input files:
  PM25:                139,168 rows
  GEE Features:        186,013 rows

Output file (combined_gee_pm25.csv):
  Rows:                83,704
  Columns:             43

Data coverage:
  Date range:          2024-05-16 to 2026-05-01
  Unique dates:        715
  Unique locations:    423

Missing values per column:
  ✓ No missing values!

✓ Merge completed successfully!


## Sample Data Inspection

In [15]:
print("Sample rows from combined dataset:\n")
print(merged_df.sample(min(5, len(merged_df))).to_string())

print("\n\nData info:")
print(merged_df.info())

Sample rows from combined dataset:

            date   id  latitude  longitude       AOD      country location_name          timezone  last_updated_epoch  temperature_celsius  temperature_fahrenheit      condition_text  wind_mph  wind_kph  wind_degree wind_direction  pressure_mb  pressure_in  precip_mm  precip_in  humidity  cloud  feels_like_celsius  feels_like_fahrenheit  visibility_km  visibility_miles  uv_index  gust_mph  gust_kph  air_quality_Carbon_Monoxide  air_quality_Ozone  air_quality_Nitrogen_dioxide  air_quality_Sulphur_dioxide    pm25  air_quality_PM10  air_quality_us-epa-index  air_quality_gb-defra-index   sunrise    sunset  moonrise   moonset       moon_phase  moon_illumination
6228  2024-07-05  318   39.5500    27.6200  0.340000       Turkey         Yaren   Europe/Istanbul          1720183500                 31.2                    88.2       Partly cloudy      16.1      25.9           20            NNE       1011.0        29.85       0.00        0.0        36     50    

In [16]:
# unique lat and log 
df_unique_locations = merged_df[['latitude', 'longitude']].drop_duplicates()
print(f"\nUnique locations (latitude, longitude): {len(df_unique_locations)}")


Unique locations (latitude, longitude): 423
